# Integration - Model Training on Reduced Embeddings
*Evaluation of classification models trained on low-dimensional embeddings of Heart Failure Prediction Dataset*  

---

## Table of Contents
1. [Setup](#setup)  
    1.2 [Imports and Reproducibility](#imports-and-reproducibility)  
    1.2 [Data Loading](#data-loading)  
    1.3 [Train Test Split](#split)  

2. [Model Training on Dimensionality-Reduced Feature Spaces](#model-on-dim-red)  
    2.1 [Model Loading](#model-loading)  
    2.2 [Dimensionality Reduction Spaces and Transformations](#dim-red-loading)  
    
3. [Results](#results)   
    3.1 [Ranking - TOP 10 Configurations](#ranking)  
    3.2 [Visualisations (3D)](#visualisations)   

4. [Conclusions](#conclusions)   

<a id="setup"></a>
## Setup
---
<a id="imports-and-reproducibility"></a>
### Imports and Reproducibility

In [25]:
import os, joblib, sys, pandas as pd, numpy as np, umap.umap_ as umap
sys.path.append(os.path.abspath(os.path.join('..')))

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from scripts.model_training.evaluation import evaluate_classification
from scripts.dim_red.plots import plot_3d

In [26]:
SEED = 42
TARGET_COL = "DEATH_EVENT"

DATA_PATH = "../outputs/eda/df_preprocessed_balanced.pkl"
MODELS_DIR = "../outputs/models/best"

<a i="data-loading"></a>
### Data loading

In [27]:
df = joblib.load(DATA_PATH)
df_copy = df.copy()

X = df_copy.iloc[:, :-1]
y = df_copy.iloc[:, -1]

TARGET_COL = 'DEATH_EVENT'

print("X shape:", X.shape)

X shape: (308, 12)


<a id="split"></a>
### Train/Test Split

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print("Split shapes:")
print("  X_train:", X_train.shape, "| X_test:", X_test.shape)

Split shapes:
  X_train: (246, 12) | X_test: (62, 12)


<a id="model-on-dim-red"></a>
## Model Training on Dimensionality-Reduced Feature Spaces
---

<a id="model-loading"></a>
### Model Loading

In [29]:
model_files = {
    "RandomForest": "randomforest.pkl",
    "AdaBoost": "adaboost.pkl",
    "XGBoost": "xgboost.pkl",
    "Naive Bayes": "naivebayes.pkl",
    "SVM": "svm.pkl",
    "Logistic Regression": "logisticregression.pkl",
    "Stacking": "STACKING.pkl",
    "Voting": "VOTING.pkl",
}

models = {
    name: clone(joblib.load(os.path.join(MODELS_DIR, filename)))
    for name, filename in model_files.items()
}

<a id="dim-red-loading"></a>
### Dimensionality Reduction Spaces and Transformations

#### Fit transformers

3 spaces are created:
  - `PCA(3)` → 3 principal components.
  - `UMAP(3)` → 3 UMAP components.
  - `MIX` → 2 UMAP components + 1 LDA component (stacked into 3 columns).

In [30]:
transformers = {}

transformers["PCA(3)"] = PCA(n_components=3, random_state=SEED).fit(X_train)
transformers["UMAP(3)"] = umap.UMAP(n_components=3, n_neighbors=30, min_dist=0.1, metric="euclidean", random_state=SEED).fit(X_train)

umap2 = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, metric="euclidean", random_state=SEED).fit(X_train)
lda1  = LDA(n_components=1).fit(X_train, y_train)
transformers["MIX(UMAP1,UMAP2,LDA1)"] = (umap2, lda1)

d:\OneDrive - Politechnika Łódzka\Dokumenty\TUL\semester7\inzynierka\cardio-risk-prediction\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

d:\OneDrive - Politechnika Łódzka\Dokumenty\TUL\semester7\inzynierka\cardio-risk-prediction\venv\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



#### Transform space

Baseline was added in order to compare whether dimensionally reducted spaces perform better than raw features or not. 

In [31]:
def transform_space(name, X_input):
    if name == "Baseline":
        return X_input.to_numpy()
    if name == "PCA(3)":
        return transformers["PCA(3)"].transform(X_input)
    if name == "UMAP(3)":
        return transformers["UMAP(3)"].transform(X_input)
    if name == "MIX(UMAP1,UMAP2,LDA1)":
        umap2, lda1 = transformers["MIX(UMAP1,UMAP2,LDA1)"]
        return np.column_stack([umap2.transform(X_input)[:, 0], umap2.transform(X_input)[:, 1], lda1.transform(X_input)[:, 0]])
    raise KeyError(name)

spaces = ["Baseline", "PCA(3)", "UMAP(3)", "MIX(UMAP1,UMAP2,LDA1)"]

<a id="results"></a>
## Results
---

<a id="ranking"></a>
### Ranking - TOP 10 Configurations

In [32]:
rows = []
for space in spaces:
    X_train_space = transform_space(space, X_train)
    X_test_space  = transform_space(space, X_test)

    for model_name, est in models.items():
        est_ = clone(est)
        est_.fit(X_train_space, y_train)

        df_metrics, _details = evaluate_classification(est_, X_test_space, y_test, show=False)
        m = dict(zip(df_metrics["Metric"], df_metrics["Value"]))

        rows.append({
            "Embedding | Model": f"{space} | {model_name}",
            "accuracy": float(m.get("Accuracy", np.nan)),
            "balanced accuracy": float(m.get("Balanced accuracy", np.nan)),
            "f1": float(m.get("F1", np.nan)),
            "roc_auc": float(m.get("ROC-AUC", np.nan)),
        })

ranking = pd.DataFrame(rows).sort_values(["f1"], ascending=False, ignore_index=True)
ranking.head(10)


,Embedding | Model,accuracy,balanced accuracy,f1,roc_auc
0,Baseline | Stacking,0.9355,0.9396,0.9091,0.9803
1,"MIX(UMAP1,UMAP2,LDA1) | Voting",0.9355,0.9280,0.9048,0.9628
2,"MIX(UMAP1,UMAP2,LDA1) | RandomForest",0.9355,0.9280,0.9048,0.9524
3,"MIX(UMAP1,UMAP2,LDA1) | Stacking",0.9194,0.9158,0.8837,0.9570
4,"MIX(UMAP1,UMAP2,LDA1) | Logistic Regression",0.9194,0.9042,0.8780,0.9617
5,Baseline | XGBoost,0.9194,0.9042,0.8780,0.9652
6,Baseline | Logistic Regression,0.9194,0.9042,0.8780,0.9652
7,Baseline | SVM,0.9032,0.8920,0.8571,0.9489
8,"MIX(UMAP1,UMAP2,LDA1) | Naive Bayes",0.9032,0.8920,0.8571,0.9512
9,Baseline | RandomForest,0.9032,0.8804,0.8500,0.9849


- The best overall result is **Baseline | Stacking** – highest accuracy, balanced accuracy, F1, and ROC AUC.
  
- Using the **MIX (UMAP + LDA)** embedding does **not improve** results over the baseline.
  
- Voting and RandomForest with MIX reach similar accuracy, but lower balanced accuracy and ROC AUC.
  
- Stacking works best **without embeddings** – adding MIX slightly hurts performance.
  
- Logistic Regression and XGBoost perform similarly in baseline and MIX setups.
  
- SVM and Naive Bayes are clearly weaker than ensemble models.
  
- **Complex embeddings are not needed** – the baseline feature space is enough.

<a id="visualisations"></a>
### Visualisations (3D)

Embeddings below are computed using transformers **fit on TRAIN** and then transformed for **ALL samples** (train+test).

Plotted are:
- true label,
- predicted risk classes,
- TP/TN/FP/FN on the TEST split.


In [33]:
best = ranking.iloc[0]
best_space, best_model = best["Embedding | Model"].split(" | ", 1)
print("Best:", best_space, "|", best_model)

Best: Baseline | Stacking


#### Refit Best Model & Prepare Visual Data

In [34]:
Xt_tr = transform_space(best_space, X_train)
Xt_te = transform_space(best_space, X_test)
Xt_all = transform_space(best_space, X)

best_est = models[best_model]
best_est.fit(Xt_tr, y_train)

_, details_all = evaluate_classification(best_est, Xt_all, y, show=False)
score_all = details_all["y_score"]

# 3D PCA for visualization only
X_vis = PCA(n_components=3).fit_transform(Xt_all)

df_best = (
    pd.DataFrame(X_vis, columns=["Component1", "Component2", "Component3"], index=X.index)
      .assign(**{
          TARGET_COL: y.to_numpy(),
          "score_all": score_all,
          "split": np.where(X.index.isin(X_train.index), "train", "test"),
      })
)

df_best.head()

,Component1,Component2,Component3,DEATH_EVENT,score_all,split
0,-1.415809,-0.598506,0.550299,0,0.368786,train
1,-1.821780,-0.535257,0.543091,0,0.742462,test
2,-1.295923,0.116354,-0.471268,0,0.086912,train
3,-0.653556,-0.706220,-0.744211,0,0.634753,train
4,1.623443,-0.103970,-0.427214,0,0.661650,train


#### Visualisation - Colour by True Label

In [35]:
plot_3d(
    df_best[["Component1", "Component2", "Component3"]],
    df_best[TARGET_COL],
    algorithm=f"{best_space} (PCA3 view)",
    data_name="BASELINE",
)

- Classes (DEATH_EVENT 0 and 1) are strongly mixed in the PCA space.
  
- There is no clear separation between outcomes in the first 3 components.

- Good model results come from **non-linear models**, not from visible clusters.


#### Visualisation - Colour by Risk Bins (Predicted Scores)

Discrete 5 score bins are defined to plot classes by colours.

In [36]:
bins = [-np.inf, 0.2, 0.4, 0.6, 0.8, np.inf]
labels = ["(0-0.2]", "(0.2-0.4]", "(0.4-0.6]", "(0.6-0.8]", "(0.8 -1.0]"]
df_best["risk_bin"] = pd.cut(df_best["score_all"], bins=bins, labels=labels)

df_best["risk_bin"].value_counts()

risk_bin
(0-0.2]       184
(0.8 -1.0]     96
(0.2-0.4]      14
(0.6-0.8]       8
(0.4-0.6]       6
Name: count, dtype: int64

In [37]:
plot_3d(
    df_best[["Component1", "Component2", "Component3"]],
    df_best["risk_bin"],
    algorithm=f"{best_space} (PCA3 view)",
    target_name="predicted risk (binned)",
    data_name="BASELINE",
)

- Plots the same 3D view (as Coloured by True Label), but it is coloured by the binned predicted risk.
  
- Different risk levels are spread across the space and strongly mixed.

- No clear regions, however low (0-0.2] and high (0.8-1.0] risk regions are appearing in multiple groups.
  
- Risk increases smoothly, not in separate clusters.
  
- The model captures risk in a **continuous way**, not as groups.

#### Visualisation - Test Set Error Visualisation

Classification Outcome - TP / TN / FP / FN labeling

In [38]:
best_est = clone(best_est)
best_est.fit(X_train_space, y_train)

df_metrics_te, details_te = evaluate_classification(best_est, X_test_space, y_test, show=False)
y_pred_te = details_te["y_pred"]

if X_test_space.shape[1] != 3:
    Xte_vis = PCA(n_components=3).fit(X_test_space)
else:
    Xte_vis = X_test_space
    
df_te = pd.DataFrame(Xte_vis, columns=["Component1", "Component2", "Component3"])
y_true_te = y_test.to_numpy(dtype=int, copy=False)

df_te[TARGET_COL] = y_true_te
df_te["pred"] = y_pred_te

codes = 2 * y_true_te + y_pred_te
df_te["err_type"] = pd.Categorical(
    np.array(["TN", "FP", "FN", "TP"], dtype=object)[codes],
    categories=["TN", "FP", "FN", "TP"],
    ordered=True,
)

In [39]:
plot_3d(
    df_te[["Component1", "Component2", "Component3"]],
    df_te["err_type"],
    algorithm=f"{best_space} (PCA3 view) + {best_model}",
    target_name="test errors",
    data_name="TEST ONLY",
)

<a id="conclusions"></a>
## Conclusions
---

- The best setup proves to be **Baseline + Stacking** (highest F1 and strong ROC AUC).

- The **MIX(UMAP1, UMAP2, LDA1)** embedding did **not** improve results compared to the baseline.
  
- When coloring by **predicted risk (binned)**, risk levels are also **spread and mixed** — risk looks continuous, not like clear clusters.

- Good performance comes mainly from the **model’s non-linear decision logic**, not from “nice” separable geometry in the embedding.
  
- Test-set error visualization (TP/TN/FP/FN) helps see where mistakes happen, but errors are not isolated into one clear area of the PCA view.

- For this dataset, **using the original feature space is enough**, and adding extra embeddings is not necessary for better classification results.
